# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_dict = dataset.metadata.to_json()
print(f"Dataset Name: {metadata_dict['name']}")
print(f"Description: {metadata_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`.

We list the available record sets and their properties. You can repeat this process for each to find available fields (`@id` of columns/fields).

In [ ]:
# List all record sets in the dataset
dataset_json = dataset.metadata.to_jsonld()
record_sets = []
if 'recordSet' in dataset_json:
    if isinstance(dataset_json['recordSet'], list):
        record_sets = dataset_json['recordSet']
    else:
        record_sets = [dataset_json['recordSet']]
else:
    from collections.abc import Iterable
    # Try to search for RecordSets in all entities
    record_sets = [n for n in dataset_json.get('@graph', []) if n.get('@type') == 'cr:RecordSet']

# If only @id references, resolve them
def find_by_id(js, target_id):
    if '@graph' in js:
        for node in js['@graph']:
            if node.get('@id') == target_id:
                return node
    return None

record_set_infos = []
if len(record_sets) > 0:
    for rso in record_sets:
        if isinstance(rso, dict):
            rs_id = rso.get('@id')
        else:
            rs_id = rso
        rs_info = find_by_id(dataset_json, rs_id)
        if rs_info is not None:
            record_set_infos.append(rs_info)
// Print all record set @ids, names and the field @ids
for idx, rs in enumerate(record_set_infos):
    print(f"RecordSet {idx+1}")
    print(f"  @id: {rs.get('@id')}")
    print(f"  name: {rs.get('schema:name', rs.get('name', 'N/A'))}")
    # Print columns (fields)
    # Look for 'field' or 'column'
    field_ids = rs.get('cr:field') or rs.get('field') or rs.get('cr:column') or rs.get('column')
    if not field_ids:
        print("  No fields found in this RecordSet.")
        continue
    if not isinstance(field_ids, list):
        field_ids = [field_ids]
    print("  Fields (by @id):")
    for fid in field_ids:
        print(f"    - {fid}")
else:
    if isinstance(record_sets, list) and len(record_sets) == 0:
        print("No record sets found in the Croissant metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found in the overview.

In [ ]:
# For illustration, get all available RecordSet @ids
available_record_set_ids = [rs.get('@id') for rs in record_set_infos]
print("RecordSet @ids:", available_record_set_ids)

# Example: Load each record set's records into DataFrames
dataframes = {}
for record_set_id in available_record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    if len(dataframes[record_set_id]) > 0:
        print(f"Fields for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for {record_set_id}.")

# Choose the first available record set for further analysis
if available_record_set_ids:
    selected_record_set_id = available_record_set_ids[0]
    print(f"Selected RecordSet for analysis: {selected_record_set_id}")
    selected_df = dataframes[selected_record_set_id]
else:
    selected_record_set_id = None
    selected_df = pd.DataFrame([])

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This example will:
- Identify a numeric field (e.g., coverage percentage, molecular weight, peptide counts)
- Filter entries where its value exceeds a threshold
- Normalize the field (Z-score)
- Optionally, group by a categorical field (e.g., description/category if present)

**Field selection uses field `@id` as required.**

In [ ]:
# Choose a numeric field for demonstration. If unsure, print all columns.
print("Available columns in selected record set:")
print(selected_df.columns.tolist())

# Heuristic: Try common field names / IDs for demonstration purposes
possible_numeric_fields = [
    'cr:coverage_percentage', 'cr:peptide_count', 'cr:molecular_weight', 'cr:MW', 'cr:abundance', 'cr:normalized_abundance',
    'coverage_percentage', 'peptide_count', 'molecular_weight', 'MW', 'abundance', 'normalized_abundance',
    'coverage', 'count', 'value', 'number',
]
numeric_field_id = None
for candidate in possible_numeric_fields:
    if candidate in selected_df.columns:
        numeric_field_id = candidate
        break
if numeric_field_id is None and len(selected_df.columns) > 0:
    # Try to pick the first numeric dtype field
    for col in selected_df.columns:
        if pd.api.types.is_numeric_dtype(selected_df[col]):
            numeric_field_id = col
            break

if numeric_field_id is not None:
    print(f"Using numeric field: {numeric_field_id}")
    # Ensure to remove missing values and incorrect types
    df_num = pd.to_numeric(selected_df[numeric_field_id], errors='coerce')
    threshold = df_num.mean() if pd.notnull(df_num.mean()) else 0
    filtered_df = selected_df[df_num > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Add normalized field
    filtered_df[numeric_field_id + '_normalized'] = (df_num - df_num.mean()) / df_num.std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Try grouping by a non-numeric field (e.g., description or category)
    possible_group_fields = [
        'cr:description', 'description', 'cr:group', 'group', 'cr:category', 'category', 'cr:protein', 'protein', 'cr:accession', 'accession'
    ]
    group_field_id = None
    for candidate in possible_group_fields:
        if candidate in selected_df.columns:
            group_field_id = candidate
            break
    if group_field_id is not None:
        print(f"Grouping by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data (mean {numeric_field_id}) by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following cell creates a histogram and boxplot for the selected numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in selected_df.columns:
    fig, axes = plt.subplots(1,2, figsize=(12,5))
    sns.histplot(pd.to_numeric(selected_df[numeric_field_id], errors='coerce').dropna(), ax=axes[0], kde=True)
    axes[0].set_title(f'Histogram of {numeric_field_id}')
    sns.boxplot(x=pd.to_numeric(selected_df[numeric_field_id], errors='coerce').dropna(), ax=axes[1])
    axes[1].set_title(f'Boxplot of {numeric_field_id}')
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field to plot.")

## 6. Conclusion
In this notebook, we demonstrated loading and initial data exploration of the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using the `mlcroissant` library.

Key steps included:
- Discovering available record sets and fields using their `@id`
- Loading records into pandas DataFrames
- Filtering and normalizing a numeric field
- Grouping and visualizing the data

Further analysis can leverage the Croissant schema to enable advanced workflows such as merging linked record sets, annotating rich metadata, or integrating with additional FAIR-compliant tools.